In [ ]:
import os
import sys
import time
import math
import copy
import shutil
import random
import tarfile
import logging
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from urllib.request import urlretrieve
from dataclasses import dataclass
from tqdm import tqdm
from typing import Optional, Dict

import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import recall_score

In [ ]:
# --- LOGGING SETUP --- #
def setup_logger(name: str = "Factory_MVTec"):
    logger = logging.getLogger(name)

    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(logging.INFO)
    logger.propagate = False #Stopping Google Colab hidden logger
    formatter = logging.Formatter('%(asctime)s - [%(levelname)s] - %(message)s', datefmt='%H:%M:%S')
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(formatter)
    logger.addHandler(ch)

    return logger

logger = setup_logger()

In [ ]:
# --- CONFIGURATION --- #
@dataclass
class InspectionConfig:
    # Data Sources
    DATA_URL: str = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937370-1629958698/bottle.tar.xz"
    ARCHIVE_NAME: str = "bottle.tar.xz"
    DATA_ROOT: Path = Path("bottle")

    # Splitting Train
    TRAIN_SPLIT_RATIO: float = 0.20

    # Training Hyperparameters
    BATCH_SIZE: int = 32 # Standard for ResNet18.
    NUM_WORKERS: int = 2
    NUM_CLASSES: int = 2
    EPOCHS: int = 10
    LEARNING_RATE: float = 1e-4
    SEED: int = 42

    # Checkpointing
    SAVE_PATH: str = "best_model_mvtec.pth"

In [ ]:
# --- THE OOP SYSTEM --- #
class BottleQualityController:
    def __init__(self, config: InspectionConfig):
        """Initializes the Quality Control System."""
        self.cfg = config
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self._set_seed()
        self._audit_hardware()

        self.train_transform = transforms.Compose([

                # Geometry Preservation
                # Resize to 224 x 224
                transforms.Resize((224,224)),

                # Industrial Augmentations (Conveyor Belt Simulation)
                # RandomAffine simulates slight vibrations/misalignments on the line.
                # - degrees=5: Slight rotation
                # - translate=(0.05, 0.05): Slight position shift
                # - scale=(0.98, 1.02): Slight camera zoom variance
                transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.98, 1.02)),

                # Sensor Noise Simulation
                # Robustness against slight lighting changes in the factory
                transforms.ColorJitter(brightness=0.1, contrast=0.1),

                # Normalization
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])

        self.val_transform = transforms.Compose([

                # Deterministic Pipeline for Evaluation
                transforms.Resize((224,224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])

        # State Placeholders
        self.model = None
        self.dataloaders = {}
        self.dataset_sizes = {}
        self.class_names = []
        self.anomaly_idx = None

    def _set_seed(self):

        """
        Configures random number generators to ensure deterministic execution.
        Essential for experiment validation and regression testing.

        Args:
            seed (int): The seed value to be used for all RNGs.
        """
        seed = self.cfg.SEED
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        logger.info(f"Global Seed locked to: {seed}")

    def _audit_hardware(self):

        """
        In production, knowing the specific hardware capabilities allows for
        dynamic batch sizing and prevents OOM (Out Of Memory) errors.

        """

        logger.info(f"Computation Device: {self.device.type.upper()}")
        if self.device.type == 'cuda':
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
            logger.info(f"   - GPU Model: {gpu_name} | VRAM: {gpu_mem:.2f} GB")

    def ingest_and_structure_data(self):
        """Handles download, extraction, and industrial restructuring."""
        root = self.cfg.DATA_ROOT

        # Download & Extract
        if not root.exists():
            logger.info("Downloading MVTec Dataset...")
            urlretrieve(self.cfg.DATA_URL, self.cfg.ARCHIVE_NAME)
            with tarfile.open(self.cfg.ARCHIVE_NAME, "r:xz") as tar:
                tar.extractall()
            os.remove(self.cfg.ARCHIVE_NAME)
            logger.info("Dataset extracted.")
        else:
            logger.info("Dataset found. Skipping download.")

        # Restructuring (MVTec to Binary Classification)

        # MVTec AD structure:
        #   train/good
        #   test/good, test/broken_large, test/contamination...
        #
        # Our Binary Classification Goal:
        #   train/good  vs  train/anomaly (needs to be created)
        #   val/good    vs  val/anomaly   (needs to be created)

        train_anomaly_dir = root / 'train' / 'anomaly'
        val_anomaly_dir = root / 'test' / 'anomaly'

        if train_anomaly_dir.exists() and val_anomaly_dir.exists():
            logger.info("Data already rebalanced. Skipping restructuring.")
            return

        logger.info("Executing Data Rebalancing Strategy...")
        os.makedirs(train_anomaly_dir, exist_ok=True)
        os.makedirs(val_anomaly_dir, exist_ok=True)

        test_dir = root / 'test'
        defect_types = [d.name for d in test_dir.iterdir() if d.is_dir() and d.name not in ['good', 'anomaly']]

        for defect in defect_types:
            source_path = test_dir / defect
            images = sorted([f for f in source_path.iterdir() if f.suffix.lower() in {'.png', '.jpg'}])

            n_train = math.ceil(len(images) * self.cfg.TRAIN_SPLIT_RATIO)

            for i, img_path in enumerate(images):
                new_name = f"{defect}_{img_path.name}"
                target_dir = train_anomaly_dir if i < n_train else val_anomaly_dir
                shutil.move(str(img_path), str(target_dir / new_name))

            try:
                os.rmdir(source_path)
            except OSError:
                pass

        logger.info("Restructuring complete. Strict binary structure enforced.")

    def prepare_dataloaders(self):

        """Creates datasets, applies industrial augmentation, and balances classes."""
        logger.info("Building Data Pipelines...")

        train_ds = datasets.ImageFolder(str(self.cfg.DATA_ROOT / 'train'), self.train_transform)
        val_ds = datasets.ImageFolder(str(self.cfg.DATA_ROOT / 'test'), self.val_transform)

        # Class Balancing Logic
        # Creates a WeightedRandomSampler to balance unbalanced datasets

        targets = torch.tensor(train_ds.targets)

        # Calculate weight per class (Inverse frequency)
        # Fewer samples = Higher weight
        class_weights = 1. / torch.bincount(targets).float()

        # Assign weight to every sample in the dataset
        sample_weights = class_weights[targets]

        # Create Sampler
        # replacement=True allows picking the same 'anomaly' image multiple times per epoch
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

        # When using a Sampler, 'shuffle' must be False in 'train' (the sampler handles the shuffling).
        # Shuffle in 'val' always False for deterministic resuslts
        self.dataloaders = {
            'train': DataLoader(train_ds,
                                batch_size=self.cfg.BATCH_SIZE,
                                sampler=sampler,
                                num_workers=self.cfg.NUM_WORKERS,
                                pin_memory= True, # Forces matrix into page-locked RAM for instant GPU transfer
                                prefetch_factor= 2, # CPU pre-loads 2 batches ahead of the GPU
                                persistent_workers= True # Prevents OS from destroying/rebuilding threads every epoch
                                ),

            'val': DataLoader(val_ds,
                              batch_size=self.cfg.BATCH_SIZE,
                              shuffle=False,
                              num_workers=self.cfg.NUM_WORKERS,
                              pin_memory=True, # Forces matrix into page-locked RAM for instant GPU transfer
                              prefetch_factor=2, # CPU pre-loads 2 batches ahead of the GPU
                              persistent_workers=True # Prevents OS from destroying/rebuilding threads every epoch
                              )
        }

        self.dataset_sizes = {'train': len(train_ds), 'val': len(val_ds)}
        self.class_names = train_ds.classes
        self.anomaly_idx = train_ds.class_to_idx['anomaly']

        logger.info(f"Pipelines Ready. Classes: {self.class_names}")

    def build_model(self):
        """Initializes the ResNet18 architecture for AOI."""

        logger.info("Initializing ResNet-18 Backbone...")

        is_gpu = torch.cuda.is_available()
        self.scaler = torch.amp.GradScaler('cuda' if is_gpu else 'cpu', enabled=is_gpu)

        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.model = models.resnet18(weights=weights)

        # Freeze Feature Extractor
        for param in self.model.parameters():
            param.requires_grad = False

        # Replace Classification Head
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, self.cfg.NUM_CLASSES)
        self.model = self.model.to(self.device)

        # Optimization Setup
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.fc.parameters(), lr=self.cfg.LEARNING_RATE)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=0.1, patience=3)
        logger.info("Architecture setup complete.")

    def train(self):
        """Executes the training loop with Recall-based checkpointing."""
        if not self.model: self.build_model()

        since = time.time()
        best_model_wts = copy.deepcopy(self.model.state_dict())
        best_acc, best_recall = 0.0, 0.0

        logger.info(f"Commencing Factory Training Protocol ({self.cfg.EPOCHS} Epochs)...")

        for epoch in range(self.cfg.EPOCHS):
            logger.info(f"--- Epoch {epoch + 1}/{self.cfg.EPOCHS} ---")

            for phase in ['train', 'val']:
                self.model.train() if phase == 'train' else self.model.eval()

                running_loss, running_corrects = 0.0, 0
                all_preds, all_labels = [], []

                pbar = tqdm(self.dataloaders[phase], desc=phase.upper(), leave=False)
                for inputs, labels in pbar:
                    inputs, labels = inputs.to(self.device), labels.to(self.device)
                    self.optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):

                        # Cast forward pass to FP16 to save VRAM
                        with torch.amp.autocast(device_type = self.device.type):

                            outputs = self.model(inputs)
                            _, preds = torch.max(outputs, 1)
                            loss = self.criterion(outputs, labels)

                            if phase == 'train':
                                # Scale the loss to prevent FP16 gradient vanishing to zero
                                self.scaler.scale(loss).backward()
                                self.scaler.step(self.optimizer)
                                self.scaler.update()
                                #loss.backward()
                                #self.optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
                    pbar.set_postfix(loss=loss.item())

                epoch_loss = running_loss / self.dataset_sizes[phase]
                epoch_acc = running_corrects.double() / self.dataset_sizes[phase]
                epoch_recall = recall_score(all_labels, all_preds, pos_label=self.anomaly_idx, zero_division=0)

                logger.info(f"{phase.upper()} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Anomaly Recall: {epoch_recall:.4f}")

                if phase == 'val':
                    self.scheduler.step(epoch_loss)

                    # Tie-Breaker Logic
                    save_model = False
                    if epoch_acc > best_acc:
                        save_model, reason = True, "Accuracy Jump"
                    elif epoch_acc == best_acc and epoch_recall > best_recall:
                        save_model, reason = True, "Recall Tie-Break"

                    if save_model:
                        best_acc, best_recall = epoch_acc, epoch_recall
                        best_model_wts = copy.deepcopy(self.model.state_dict())
                        self.save_model(self.cfg.SAVE_PATH)
                        logger.info(f"NEW ARTIFACT SAVED! ({reason})")

        time_elapsed = time.time() - since
        logger.info(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
        self.model.load_state_dict(best_model_wts)

    def visualize_predictions(self, num_images=6, save_path="inspection_grid.png"):
        """
        Qualitative Analysis Tool.
        Generates a visual grid of Validation predictions vs Ground Truth.
        """
        if not self.model:
            logger.error("Model not loaded. Cannot visualize.")
            return

        logger.info(f"Generating visual inspection grid for {num_images} samples...")
        was_training = self.model.training
        self.model.eval()
        images_so_far = 0

        # Dynamic Grid Layout
        cols = 3
        rows = math.ceil(num_images / cols)
        plt.figure(figsize=(15, 5 * rows))

        with torch.no_grad():
            for i, (inputs, labels) in enumerate(self.dataloaders['val']):
                inputs = inputs.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(inputs)
                _, preds = torch.max(outputs, 1)

                for j in range(inputs.size()[0]):
                    images_so_far += 1
                    ax = plt.subplot(rows, cols, images_so_far)
                    ax.axis('off')

                    # Denormalize for human viewing
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img = inputs.cpu().data[j].numpy().transpose((1, 2, 0))
                    img = std * img + mean
                    img = np.clip(img, 0, 1)

                    # Industrial Color Coding (Green = Pass/Match, Red = Fail/Mismatch)
                    is_correct = (preds[j] == labels[j])
                    color = 'green' if is_correct else 'red'

                    title = f"Pred: {self.class_names[preds[j]]}\nReal: {self.class_names[labels[j]]}"
                    ax.set_title(title, color=color, fontweight='bold')
                    ax.imshow(img)

                    if images_so_far == num_images:
                        self.model.train(mode=was_training)
                        plt.tight_layout()
                        plt.savefig(save_path)
                        logger.info(f"Visual inspection saved to: {save_path}")
                        plt.show()
                        return

        self.model.train(mode=was_training)

    def save_model(self, path: str):
        torch.save(self.model.state_dict(), path)

    def load_model(self, path: str):

        if not os.path.exists(path): raise FileNotFoundError(f"Model file {path} missing.")
        if not self.model: self.build_model()
        self.model.load_state_dict(torch.load(path, map_location=self.device))
        logger.info(f"Artifact loaded from {path}")

    def predict_image(self, image_path: str) -> str:

        """Production inference for API."""

        img = Image.open(image_path).convert('RGB')
        preprocess = transforms.Compose([
            transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        img_tensor = preprocess(img).unsqueeze(0).to(self.device)

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(img_tensor)
            _, preds = torch.max(outputs, 1)

        return self.class_names[preds[0]]

In [ ]:
# ---SMOKE TEST --- #
def run_smoke_test(controller: BottleQualityController):
    logger.info("Initiating Smoke Test...")
    if controller.model is None: return False
    try:
        dummy_input = torch.randn(1, 3, 224, 224).to(controller.device)
        controller.model.eval()
        with torch.no_grad():
            output = controller.model(dummy_input)
        if output.shape == (1, 2):
            logger.info("Smoke Test Passed: Network ready for traffic.")
            return True
        return False
    except Exception as e:
        logger.error(f"Smoke Test Failed: {e}")
        return False

In [ ]:
# --- EXECUTION --- #

if __name__ == "__main__":
    config = InspectionConfig(EPOCHS=20) # Set low for initial testing
    aoi_system = BottleQualityController(config)

    # --- FACTORY PHASE (Run locally/GPU once) ---
    aoi_system.ingest_and_structure_data()
    aoi_system.prepare_dataloaders()
    aoi_system.train()

    # --- DEPLOYMENT PHASE (Simulating Docker/API startup) ---
    prod_system = BottleQualityController(config)
    prod_system.prepare_dataloaders() # Needed to get class names mapped
    prod_system.load_model(config.SAVE_PATH)

    if run_smoke_test(prod_system):
        print("[SYSTEM SECURED] Ready for Docker Integration.")

        # --- Visual Verification ---
        #prod_system.visualize_predictions(num_images=6)